In [1]:
!pip uninstall -y torchvision torchaudio transformers 2>/dev/null
!pip install -U spacy spacy-transformers
!pip install "transformers==4.46.3" "huggingface-hub==0.26.5"

import torch
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.2/33.2 MB 62.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.5/780.5 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.4/313.4 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 93.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 107.9 MB/s eta 0:00:0000:010:01
   ━━

In [2]:
from importlib.metadata import version
import torch

print("transformers:", version("transformers"))      # must be 4.46.3
print("spacy-transformers:", version("spacy-transformers"))
print("CUDA available:", torch.cuda.is_available())

# This is the exact import that was crashing — confirm it works now
from transformers import AutoModel
print("transformers import OK ✅")

transformers: 4.46.3
spacy-transformers: 1.4.0
CUDA available: True
transformers import OK ✅


In [3]:
import spacy
from spacy.tokens import DocBin

nlp = spacy.blank("xx")

def load_bio(filepath):
    sentences, tokens, tags = [], [], []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                if tokens:
                    sentences.append((tokens, tags))
                    tokens, tags = [], []
                continue
            parts = line.split()
            tokens.append(parts[0])
            tags.append(parts[1])
    if tokens:
        sentences.append((tokens, tags))
    return sentences

def bio_to_spacy(sentences):
    db = DocBin()
    for tokens, tags in sentences:
        doc = nlp.make_doc(" ".join(tokens))
        ents = []
        start = None
        label = None
        for i, tag in enumerate(tags):
            if tag.startswith("B-"):
                if start is not None:
                    ents.append((start, i, label))
                start = i
                label = tag[2:]
            elif tag.startswith("I-"):
                continue
            else:
                if start is not None:
                    ents.append((start, i, label))
                    start = None
        if start is not None:
            ents.append((start, len(tags), label))

        spans = []
        for s, e, lbl in ents:
            span = doc.char_span(
                doc[s].idx,
                doc[e - 1].idx + len(doc[e - 1]),
                label=lbl,
            )
            if span:
                spans.append(span)
        doc.ents = spans
        db.add(doc)
    return db

sentences = load_bio("/kaggle/input/datasets/tudortefanblidaru/kurmaji/adyan.txt")   # change filename if needed
docbin = bio_to_spacy(sentences)
docbin.to_disk("ner.spacy")
print(f"Converted {len(sentences)} sentences → ner.spacy")

Converted 22800 sentences → ner.spacy


In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

In [4]:
config = """
[paths]
train = ner.spacy
dev = ner.spacy

[nlp]
lang = "xx"
pipeline = ["transformer", "ner"]
batch_size = 1

[components]

[components.transformer]
factory = "transformer"

[components.transformer.model]
@architectures = "spacy-transformers.TransformerModel.v3"
name = "xlm-roberta-base"
tokenizer_config = {}
transformer_config = {}
mixed_precision = false
grad_scaler_config = {}

[components.transformer.model.get_spans]
@span_getters = "spacy-transformers.strided_spans.v1"
window = 32
stride = 16

[components.ner]
factory = "ner"

[components.ner.model]
@architectures = "spacy.TransitionBasedParser.v2"
state_type = "ner"
extra_state_tokens = false
hidden_width = 64
maxout_pieces = 2
use_upper = true
nO = null

[components.ner.model.tok2vec]
@architectures = "spacy-transformers.TransformerListener.v1"
grad_factor = 1.0
upstream = "*"
pooling = {"@layers": "reduce_mean.v1"}

[training]
seed = 0
gpu_allocator = "pytorch"
dropout = 0.1
max_epochs = 20
patience = 8000
eval_frequency = 400
frozen_components = []
annotating_components = []
dev_corpus = "corpora.dev"
train_corpus = "corpora.train"

[training.logger]
@loggers = "spacy.ConsoleLogger.v1"
progress_bar = true

[training.optimizer]
@optimizers = "Adam.v1"
beta1 = 0.9
beta2 = 0.999
L2_is_weight_decay = true
L2 = 0.01
grad_clip = 0.5
use_averages = false
eps = 0.00000001

[training.optimizer.learn_rate]
@schedules = "warmup_linear.v1"
warmup_steps = 200
total_steps = 20000
initial_rate = 2e-5

[training.batcher]
@batchers = "spacy.batch_by_words.v1"
discard_oversize = true
tolerance = 0.2

[training.batcher.size]
@schedules = "compounding.v1"
start = 50
stop = 150
compound = 1.001

[corpora]

[corpora.train]
@readers = "spacy.Corpus.v1"
path = ${paths.train}
max_length = 64

[corpora.dev]
@readers = "spacy.Corpus.v1"
path = ${paths.dev}
max_length = 64
"""

with open("config.cfg", "w") as f:
    f.write(config)
print("config.cfg written ✅")

# Verify key settings
with open("config.cfg") as f:
    for line in f:
        if any(k in line for k in ["warmup", "initial_rate", "grad_clip", "mixed"]):
            print(line.strip())

config.cfg written ✅
mixed_precision = false
grad_clip = 0.5
@schedules = "warmup_linear.v1"
warmup_steps = 200
initial_rate = 2e-5


In [5]:
import os, torch, gc
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
gc.collect()
torch.cuda.empty_cache()
!python -m spacy train config.cfg --output output --gpu-id 0

✔ Created output directory: output
ℹ Saving to output directory: output
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================
tokenizer_config.json: 100%|██████████████████| 25.0/25.0 [00:00<00:00, 214kB/s]
config.json: 100%|█████████████████████████████| 615/615 [00:00<00:00, 5.76MB/s]
sentencepiece.bpe.model: 100%|█████████████| 5.07M/5.07M [00:00<00:00, 27.7MB/s]
tokenizer.json: 9.10MB [00:00, 27.2MB/s]
2026-04-12 13:41:28.896303: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776001289.109652     136 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776001289.181115     136 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already bee

In [6]:
# Evaluate per-label scores
!python -m spacy evaluate output/model-best ner.spacy --gpu-id 0

# Zip for download
import shutil
shutil.make_archive("/kaggle/working/kurdish-ner-v1", "zip", "output/model-best")

ℹ Using GPU: 0
2026-04-12 20:05:19.688189: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776024319.717378     462 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776024319.725810     462 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776024319.758139     462 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776024319.758213     462 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776024319.758228     462 computation_placer.cc:177] computa

'/kaggle/working/kurdish-ner-v1.zip'

In [7]:
import spacy

# Load it
nlp = spacy.load("kurdish-ner-v1")

# Run on any Kurdish text
doc = nlp("کەرکووک شارێکی گەورەیە لە عێراق")

# Get entities
for ent in doc.ents:
    print(ent.text, ent.label_)
# Output:
# کەرکووک    LOC
# عێراق      COUNTRY

OSError: [E050] Can't find model 'kurdish-ner-v1'. It doesn't seem to be a Python package or a valid path to a data directory.

In [ ]:
with open("config.cfg") as f:
    for line in f:
        if "mixed" in line:
            print(line.strip())